In [ ]:
import glob
import os
import random
import numpy as np
import librosa
import soundfile as sf
from tqdm import tqdm
from pathlib import Path
from collections import defaultdict

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

TARGET_SR = 16000
SEG = TARGET_SR  # 1 second

wakeword_path = "./raw/synthetic-wakeword-wake_up/"
speech_commands_path = "./raw/Google-Speech-Commands-V2/"
wham_noise_path = "./raw/wham_noise/"
esc50_audio_path = "./raw/ESC-50-master/audio/"
local_noise_path = "./raw/my-noise/"

In [ ]:
wakeword_files = glob.glob(os.path.join(wakeword_path, "*.wav"))
speech_command_files = glob.glob(os.path.join(speech_commands_path, "**/*.wav"), recursive=True)
speech_command_files = [f for f in speech_command_files if "background" not in f]
wham_noise_files = glob.glob(os.path.join(wham_noise_path, "*.wav"))
esc50_noise_files = glob.glob(os.path.join(esc50_audio_path, "*.wav"))
local_noise_files = glob.glob(os.path.join(local_noise_path, "*.wav"))

print(f"wakeword: {len(wakeword_files)}, speech_commands: {len(speech_command_files)}, wham: {len(wham_noise_files)}, esc50: {len(esc50_noise_files)}, local_noise: {len(local_noise_files)}")


In [ ]:
# === PROCESS: resample + cắt 1s ===

def process_to_dir(file_list, out_dir, subsample_step=None, prefix_word=False):
    """
    subsample_step: chỉ dùng cho my-noise, lấy 1 clip mỗi N giây
    prefix_word: nếu True, giữ tên folder làm prefix trong tên file (cho GSC)
    """
    os.makedirs(out_dir, exist_ok=True)
    for f in tqdm(file_list):
        audio, _ = librosa.load(f, sr=TARGET_SR, mono=True)
        base = os.path.splitext(os.path.basename(f))[0]

        if prefix_word:
            word = os.path.basename(os.path.dirname(f))
            base = f"{word}_{base}"

        step = subsample_step if subsample_step else SEG
        indices = range(0, len(audio) - SEG + 1, step)

        for j, i in enumerate(indices):
            chunk = audio[i:i + SEG]
            if len(chunk) < SEG:
                continue
            out = os.path.join(out_dir, f"{base}_{j:04d}.wav")
            sf.write(out, chunk, TARGET_SR)


In [ ]:
print("Processing wakeword...")
process_to_dir(wakeword_files, "./temp/pos/")

In [ ]:
print("Processing speech commands...")
process_to_dir(speech_command_files, "./temp/neg/", prefix_word=True)


In [ ]:
print("Processing WHAM noise...")
process_to_dir(wham_noise_files, "./temp/noise1/")

print("Processing ESC-50 noise...")
process_to_dir(esc50_noise_files, "./temp/noise2/")

print("Processing local noise, subsample 1 clip / 5s...")
process_to_dir(local_noise_files, "./temp/noise3/", subsample_step=TARGET_SR * 3)

In [ ]:
wakeword_clips = glob.glob("./temp/pos/*.wav")
speech_command_clips = glob.glob("./temp/neg/*.wav")
wham_noise_clips = glob.glob("./temp/noise1/*.wav")
esc50_noise_clips = glob.glob("./temp/noise2/*.wav")
local_noise_clips = glob.glob("./temp/noise3/*.wav")

print(f"wakeword: {len(wakeword_clips)}, speech_commands: {len(speech_command_clips)}, wham: {len(wham_noise_clips)}, esc50: {len(esc50_noise_clips)}, local_noise: {len(local_noise_clips)}")


In [ ]:
# === SPLIT ESC-50 theo class (35 train / 15 OOD) ===

def esc50_class(path):
    name = os.path.splitext(os.path.basename(path))[0]
    return int(name.split('-')[-1])

esc50_by_class = defaultdict(list)
for f in esc50_noise_clips:
    esc50_by_class[esc50_class(f)].append(f)

all_classes = sorted(esc50_by_class.keys())
random.shuffle(all_classes)
ood_classes   = set(all_classes[:15])
train_classes = set(all_classes[15:])

esc50_train = [f for c in train_classes for f in esc50_by_class[c]]
esc50_ood   = [f for c in ood_classes   for f in esc50_by_class[c]]

print(f"ESC-50 train classes: {len(train_classes)} ({len(esc50_train)} clips)")
print(f"ESC-50 OOD classes:   {len(ood_classes)}   ({len(esc50_ood)} clips)")


In [ ]:
# === SPLIT: 80% train+val / 20% test, rồi train+val → 80% train / 20% val ===

def split_train_test(lst):
    lst = lst.copy()
    random.shuffle(lst)
    n = len(lst)
    t = int(n * 0.8)
    return lst[:t], lst[t:]

def split_train_val(lst):
    lst = lst.copy()
    random.shuffle(lst)
    n = len(lst)
    t = int(n * 0.8)
    return lst[:t], lst[t:]

wakeword_trainval, wakeword_test = split_train_test(wakeword_clips)
wakeword_train, wakeword_val = split_train_val(wakeword_trainval)

wham_trainval, wham_test = split_train_test(wham_noise_clips)
wham_train, wham_val = split_train_val(wham_trainval)

local_trainval, local_test = split_train_test(local_noise_clips)
local_train, local_val = split_train_val(local_trainval)

# neg: cap mỗi word trước rồi mới split
speech_commands_by_word = defaultdict(list)
for f in speech_command_clips:
    word = os.path.basename(f).split('_')[0]
    speech_commands_by_word[word].append(f)

print(f"GSC words: {len(speech_commands_by_word)}")
print(f"  min/max per word: {min(len(v) for v in speech_commands_by_word.values())} / {max(len(v) for v in speech_commands_by_word.values())}")

CAP_PER_WORD = 150
speech_commands_capped = []
for word, files in speech_commands_by_word.items():
    random.shuffle(files)
    speech_commands_capped.extend(files[:CAP_PER_WORD])

speech_commands_trainval, speech_commands_test = split_train_test(speech_commands_capped)
speech_commands_train, speech_commands_val = split_train_val(speech_commands_trainval)

print(f"speech_commands after cap: {len(speech_commands_capped)}")
print(f"  train: {len(speech_commands_train)}, val: {len(speech_commands_val)}, test: {len(speech_commands_test)}")


In [ ]:
# === AUGMENT UTILS ===

GAIN_MIN, GAIN_MAX = 0.5, 2.0
SNR_WAKEUP  = [10, 12, 15, 20]
SNR_UNKNOWN = [-5, 0, 5, 10, 15, 20]

def load_clip(path):
    audio, _ = librosa.load(path, sr=TARGET_SR, mono=True)
    if len(audio) < SEG:
        audio = np.pad(audio, (0, SEG - len(audio)))
    return audio[:SEG]

def load_clip_random_offset(path):
    audio, _ = librosa.load(path, sr=TARGET_SR, mono=True)
    if len(audio) <= SEG:
        if len(audio) < SEG:
            audio = np.pad(audio, (0, SEG - len(audio)))
        return audio[:SEG]
    offset = random.randint(0, len(audio) - SEG)
    return audio[offset:offset + SEG]

def apply_gain(audio):
    gain = random.uniform(GAIN_MIN, GAIN_MAX)
    audio = audio * gain
    max_val = np.max(np.abs(audio))
    if max_val > 1.0:
        audio = audio / max_val
    return audio

def mix_snr(speech, noise_audio, snr_db):
    speech_power = np.mean(speech ** 2) + 1e-9
    noise_power  = np.mean(noise_audio ** 2) + 1e-9
    scale = np.sqrt(speech_power / (10 ** (snr_db / 10)) / noise_power)
    mixed = speech + noise_audio * scale
    max_val = np.max(np.abs(mixed))
    if max_val > 1.0:
        mixed = mixed / max_val
    return mixed

def save_clip(audio, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    sf.write(path, audio, TARGET_SR)


In [ ]:
def generate_class(file_list, noise_pool, snr_list, out_dir, n_target):
    """
    100% gain + noise augment, không có clean hay vol-only
    """
    os.makedirs(out_dir, exist_ok=True)

    pool = file_list.copy()
    random.shuffle(pool)

    def get(idx):
        return load_clip(pool[idx % len(pool)])

    for i in range(n_target):
        audio = get(i)
        audio = apply_gain(audio)
        noise_audio = load_clip_random_offset(random.choice(noise_pool))
        snr = random.choice(snr_list)
        audio = mix_snr(audio, noise_audio, snr)
        save_clip(audio, os.path.join(out_dir, f"{i:05d}.wav"))

    print(f"  {out_dir}: {n_target} files (100% gain+noise)")

In [ ]:
dataset_root = "./dataset/"
os.makedirs(dataset_root, exist_ok=True)

wakeup_noise_pool = wham_train + local_train
unknown_noise_pool = wham_train + esc50_train


In [ ]:
# === TRAIN: 9600 total = wakeup 2400 + unknown 4800 + noise 2400 ===
print("=== TRAIN ===")

generate_class(
    file_list   = wakeword_train,
    noise_pool  = wakeup_noise_pool,
    snr_list    = SNR_WAKEUP,
    out_dir     = dataset_root + "train/wakeup/",
    n_target    = 2400,
)

generate_class(
    file_list   = speech_commands_train,
    noise_pool  = unknown_noise_pool,
    snr_list    = SNR_UNKNOWN,
    out_dir     = dataset_root + "train/unknown/",
    n_target    = 4800,
)

noise_train_pool = wham_train + esc50_train + local_train
random.shuffle(noise_train_pool)
os.makedirs(dataset_root + "train/noise/", exist_ok=True)
for i, f in enumerate(tqdm(noise_train_pool[:2400], desc="noise train")):
    save_clip(load_clip_random_offset(f), dataset_root + f"train/noise/{i:05d}.wav")


In [ ]:
# === VAL: 2400 total = wakeup 600 + unknown 1200 + noise 600 ===
print("=== VAL ===")

generate_class(
    file_list   = wakeword_val,
    noise_pool  = wakeup_noise_pool,
    snr_list    = SNR_WAKEUP,
    out_dir     = dataset_root + "val/wakeup/",
    n_target    = 600,
)

generate_class(
    file_list   = speech_commands_val,
    noise_pool  = unknown_noise_pool,
    snr_list    = SNR_UNKNOWN,
    out_dir     = dataset_root + "val/unknown/",
    n_target    = 1200,
)

noise_val_pool = wham_val + local_val
random.shuffle(noise_val_pool)
os.makedirs(dataset_root + "val/noise/", exist_ok=True)
for i, f in enumerate(tqdm(noise_val_pool[:600], desc="noise val")):
    save_clip(load_clip_random_offset(f), dataset_root + f"val/noise/{i:05d}.wav")


In [ ]:
# === TEST: 3000 total = wakeup 750 + unknown 1500 + noise 750 ===
print("=== TEST ===")

wakeup_test_noise_pool = wham_test + local_test + esc50_ood  # esc50_ood acts as fallback OOD noise
unknown_test_noise_pool = wham_test + esc50_ood + local_test

generate_class(
    file_list  = wakeword_test,
    noise_pool = wakeword_test_noise_pool,
    snr_list   = SNR_WAKEUP,
    out_dir    = dataset_root + "test/wakeup/",
    n_target   = 750,
)

generate_class(
    file_list  = speech_commands_test,
    noise_pool = unknown_test_noise_pool,
    snr_list   = SNR_UNKNOWN,
    out_dir    = dataset_root + "test/unknown/",
    n_target   = 1500,
)

noise_test_pool = esc50_ood + wham_test + local_test
random.shuffle(noise_test_pool)
os.makedirs(dataset_root + "test/noise/", exist_ok=True)
for i, f in enumerate(tqdm(noise_test_pool[:750], desc="noise test")):
    save_clip(load_clip_random_offset(f), dataset_root + f"test/noise/{i:05d}.wav")

In [ ]:
# === SUMMARY ===
print("\n=== Dataset Summary ===")
total = 0
for split in ["train", "val", "test"]:
    split_total = 0
    for cls in ["wakeup", "unknown", "noise"]:
        d = dataset_root + f"{split}/{cls}/"
        if os.path.exists(d):
            n = len(glob.glob(d + "*.wav"))
            print(f"  {split}/{cls}: {n}")
            split_total += n
    print(f"  → {split} total: {split_total}")
    total += split_total
print(f"\n  GRAND TOTAL: {total}")